# JET-MLIPs with LAMMPS

Run this notebook from `examples/lammps/`, in the env of one family, after LAMMPS is built there
(`bash lammps/build_lammps.sh`, README LAMMPS step 1). The inputs in this folder run NVT at 300 K on an equilibrated
EC/EMC box (`ecemc_9520.data`, 9,520 atoms, types C H O, which every model listed below covers).

Set `FAMILY`, and `RUN` (the models of that family to deploy and run; the table below lists every supported model).

In [ ]:
FAMILY = 'mace'   # 'sevenn', 'mace', 'nequip', 'nequix' or 'equflash'
RUN = None        # None: the first model of FAMILY below; or a list of labels, e.g. ['small', 'off-small']

## Every supported model

`label: (<model>, deploy options, output file)`. `<model>` is what the official package accepts: a pretrained name,
a nequip.net id or a checkpoint file (a fine-tuned checkpoint works the same way). Files to get first:

- SevenNet 7net-nano: the checkpoint file `checkpoint_7net_nano_6.0.pth` (not a pretrained name in sevenn 0.13.0).
- Nequix: `nequix-mp-1.pt` / `nequix-oam-1.pt`, which the nequix package downloads to `~/.cache/nequix/models/`
  (e.g. after `NequixCalculator('nequix-mp-1', backend='torch')` once).
- EquFlash: `equflash_oam_30M.pt` (v1) / `equflashv2_oam.pt` (V2) from the GGNN release.

Multi-fidelity SevenNet models need `--modal`: 7net-mf-0 `PBE` or `R2SCAN`; 7net-omni `mpa`, `omat24`,
`matpes_pbe`, `matpes_r2scan`, `mp_r2scan`, `oc20`, `oc22`, `odac23`, `spice`, `qcml`, `omol25_low`, `omol25_high`,
`pet_mad`. NequIP outputs must end with `.nequip.lmp.pt`.

In [ ]:
MODELS = {
    'sevenn': {
        '7net-nano':  ('checkpoint_7net_nano_6.0.pth', '',             '7net-nano.mliap.pt'),
        '7net-0':     ('7net-0',                       '',             '7net-0.mliap.pt'),
        '7net-mf-0':  ('7net-mf-0',                    '--modal PBE',  '7net-mf-0_pbe.mliap.pt'),
        '7net-omni':  ('7net-omni',                    '--modal mpa',  '7net-omni_mpa.mliap.pt'),
    },
    'mace': {   # MACE-MP-0 small / medium / large, MP-0b3, MPA-0, OMAT-0, MATPES, MACE-OFF23
        'small':                ('small',                '', 'small_jet.pt'),
        'medium':               ('medium',               '', 'medium_jet.pt'),
        'large':                ('large',                '', 'large_jet.pt'),
        'medium-0b3':           ('medium-0b3',           '', 'medium-0b3_jet.pt'),
        'medium-mpa-0':         ('medium-mpa-0',         '', 'medium-mpa-0_jet.pt'),
        'small-omat-0':         ('small-omat-0',         '', 'small-omat-0_jet.pt'),
        'medium-omat-0':        ('medium-omat-0',        '', 'medium-omat-0_jet.pt'),
        'mace-matpes-pbe-0':    ('mace-matpes-pbe-0',    '', 'matpes-pbe-0_jet.pt'),
        'mace-matpes-r2scan-0': ('mace-matpes-r2scan-0', '', 'matpes-r2scan-0_jet.pt'),
        'off-small':            ('off-small',            '', 'off-small_jet.pt'),
        'off-medium':           ('off-medium',           '', 'off-medium_jet.pt'),
        'off-large':            ('off-large',            '', 'off-large_jet.pt'),
    },
    'nequip': {
        'OAM-S':  ('nequip.net:mir-group/NequIP-OAM-S:0.1',  '', 'oam-s.nequip.lmp.pt'),
        'OAM-M':  ('nequip.net:mir-group/NequIP-OAM-M:0.1',  '', 'oam-m.nequip.lmp.pt'),
        'OAM-L':  ('nequip.net:mir-group/NequIP-OAM-L:0.1',  '', 'oam-l.nequip.lmp.pt'),
        'OAM-XL': ('nequip.net:mir-group/NequIP-OAM-XL:0.1', '', 'oam-xl.nequip.lmp.pt'),
        'MP-L':   ('nequip.net:mir-group/NequIP-MP-L:0.1',   '', 'mp-l.nequip.lmp.pt'),
    },
    'nequix': {
        'nequix-mp-1':  ('~/.cache/nequix/models/nequix-mp-1.pt',  '', 'nequix-mp-1.mliap.pt'),
        'nequix-oam-1': ('~/.cache/nequix/models/nequix-oam-1.pt', '', 'nequix-oam-1.mliap.pt'),
    },
    'equflash': {
        'EquFlash-OAM-30M': ('equflash_oam_30M.pt', '', 'equflash_oam_30M.mliap.pt'),
        'EquFlashV2-OAM':   ('equflashv2_oam.pt',   '', 'equflashv2_oam.mliap.pt'),   # runs without the CUDA graph
    },
}
run = RUN or list(MODELS[FAMILY])[:1]
for label, (model, opts, out) in MODELS[FAMILY].items():
    print(('* ' if label in run else '  ') + f'python -m jet_mlip.deploy {FAMILY} {model} {out} {opts}'.rstrip())

In [ ]:
import os, subprocess

def sh(cmd, env=None):
    """Run a shell command here and show the end of its output."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env={**os.environ, **(env or {})})
    print('\n'.join((r.stdout + r.stderr).splitlines()[-15:]))
    r.check_returncode()

## 1. Deploy the models to one file each

Other options (`python -m jet_mlip.deploy <family> --help`): `--radial_table N` (grid points of the tabulated
radial weights), MACE `--head NAME`.

In [ ]:
for label in run:
    model, opts, out = MODELS[FAMILY][label]
    sh(f'python -m jet_mlip.deploy {FAMILY} {model} {out} {opts}')

## 2. Run on one GPU

`in.mliap` uses `pair_style jet/mliap <file>` (= `mliap unified <file> 0`), which needs KOKKOS: `-k on g 1 -sf kk`.

In [ ]:
KK = '-k on g 1 -sf kk -pk kokkos newton on neigh half'
for label in run:
    out = MODELS[FAMILY][label][2]
    sh(f'lmp {KK} -in in.mliap -var model {out} -log log.{label}')

## 3. With D3 dispersion

`in.mliap_d3` overlays `jet/d3 9000 1600 damp_bj pbe` on the model (`pair_style hybrid/overlay`); under `-sf kk` it
runs as `jet/d3/kk`. Precision (environment variable): `JET_D3_FP64=0` on consumer GeForce cards, the default
(`JET_D3_FP64=1`) on data-center GPUs.

In [ ]:
D3_ENV = {}   # GeForce: {'JET_D3_FP64': '0'}
for label in run:
    out = MODELS[FAMILY][label][2]
    sh(f'lmp {KK} -in in.mliap_d3 -var model {out} -log log.{label}_d3', env=D3_ENV)

## 4. Several GPUs (experimental)

One node, one GPU per rank, the same deployed file. Worth it from ~50k atoms per GPU; this 9,520-atom box only shows
the command.

In [ ]:
N = 2
out = MODELS[FAMILY][run[0]][2]
sh(f'mpirun -np {N} lmp -k on g {N} -sf kk -pk kokkos newton on neigh half gpu/aware off -in in.mliap -var model {out} -log log.np{N}')

## 5. SevenNet's own pair style `jet/e3gnn`

For SevenNet models `jet/e3gnn` (= SevenNet's `e3gnn`) is faster than ML-IAP at every size measured. It needs
LAMMPS built with `SEVENN=1 bash lammps/build_lammps.sh` and runs without KOKKOS. `in.sevenn_d3` adds D3.
`sevenn get_model` takes the same `<model>` and `-m <modal>`.

In [ ]:
if FAMILY == 'sevenn':
    for label in run:
        model, opts, _ = MODELS['sevenn'][label]
        modal = opts.replace('--modal', '-m')
        sh(f'sevenn get_model {model} {modal} --enable_jet -o {label}_e3gnn.pt')
        sh(f'lmp -in in.sevenn_d3 -var model {label}_e3gnn.pt -log log.{label}_e3gnn_d3')